# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrabansal10/FlyRank_Internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: Growing pages are longer, younger, and slightly better positioned than declining pages

- **What the paper reports:** Growing content averages about 3.2K words and 184 days old, while declining content averages about 2.3K words and 230 days old. The paper describes this as an observational comparison between 74,187 rising pages and 45,272 falling pages.
- **Where the label or outcome comes from:** Trend direction is calculated from the last 30 days of impressions compared with the previous 30 days. “Up” means more than 10% growth, and “down” means more than 10% decline.
- **Does the validation design support the claim?** It supports the descriptive claim that the rising and declining cohorts differ in this portfolio. It does not support a causal claim that increasing word count or changing page age will cause growth, and it is not a future-time prediction validation.
- **Methodology question:** Could a later time-based validation test whether age, depth, and visibility measured before a 30-day outcome window predict later momentum, rather than only describing pages grouped by their already-observed trend?

### Finding 2: Recently refreshed older pages have higher health scores and impressions

- **What the paper reports:** The paper reports that content older than 365 days and refreshed within 30 days had a 3.2x higher health score and 57x more impressions than the comparison group. It presents refresh timing as a strong measured lever.
- **Where the label or outcome comes from:** This is not a supervised model label. It is a direct aggregate comparison using the local active-content feature-vector sample, with health score and impressions measured in the current snapshot.
- **Does the validation design support the claim?** It supports an observed association between recent refresh status and stronger current performance. It does not establish that refreshing caused the higher health score or impressions, because pages selected for refresh may already differ in visibility, quality, business priority, or age.
- **Methodology question:** Could the analysis compare refreshed pages with similar untreated pages using pre-refresh performance and a strictly later follow-up window? This would reduce selection bias and better test whether refreshes are followed by improvement.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TOP_K = 50

df = pd.read_csv("https://raw.githubusercontent.com/rudrabansal10/FlyRank_Internship/43b468d73eba109085f02d01f3a59754d5356453/data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

numeric_features = [
    "search_volume", "competition", "cpc",
    "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
]

X = df[numeric_features + categorical_features].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].copy()

# Position 0 means no position data.
X["has_position_data"] = (X["avg_position"] > 0).astype(int)
X["avg_position"] = X["avg_position"].replace(0, np.nan)

for column in [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
]:
    X[f"log_{column}"] = np.log1p(X[column].clip(lower=0))

numeric_features = numeric_features + [
    "has_position_data",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
]

def precision_at_k(labels, scores, k=50):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

def make_model(numeric_columns):
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_columns),
            ("categorical", categorical_pipeline, categorical_features),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE,
            )),
        ]
    )

In [2]:
# Less realistic comparison: random pages can share clients across train and test.
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

random_model = make_model(numeric_features)
random_model.fit(X_train_random, y_train_random)

random_scores = random_model.predict_proba(X_test_random)[:, 1]
random_p50 = precision_at_k(y_test_random, random_scores, TOP_K)

# Honest primary evaluation: clients never overlap.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_index, test_index = next(
    splitter.split(X, y=y, groups=groups)
)

X_train_grouped = X.iloc[train_index]
X_test_grouped = X.iloc[test_index]
y_train_grouped = y.iloc[train_index]
y_test_grouped = y.iloc[test_index]

grouped_model = make_model(numeric_features)
grouped_model.fit(X_train_grouped, y_train_grouped)

grouped_scores = grouped_model.predict_proba(X_test_grouped)[:, 1]
grouped_p50 = precision_at_k(y_test_grouped, grouped_scores, TOP_K)

validation_results = pd.DataFrame([
    {
        "split": "Random row split — diagnostic only",
        "held_out_base_rate": y_test_random.mean(),
        "precision_at_50": random_p50,
    },
    {
        "split": "Grouped client split — primary result",
        "held_out_base_rate": y_test_grouped.mean(),
        "precision_at_50": grouped_p50,
    },
])

display(validation_results.round(3))

assert set(groups.iloc[train_index]).isdisjoint(set(groups.iloc[test_index]))

,split,held_out_base_rate,precision_at_50
0,Random row split — diagnostic only,0.542,0.90
1,Grouped client split — primary result,0.511,0.84


The grouped client split is the primary result because it tests the model on clients not seen in training. The random split is diagnostic only; it may be optimistic because pages from the same client can share hidden characteristics.

The grouped result still uses a contemporaneous starter proxy label. It is stronger than a random split for client generalization, but it is not equivalent to future-time validation.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
excluded_features = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "provider_used",
    "model_used",
]

safe_feature_overlap = sorted(set(excluded_features).intersection(X.columns))

print("Excluded fields found in safe feature frame:", safe_feature_overlap)

assert safe_feature_overlap == [], "Leakage audit failed: an excluded field entered X."

# Deliberate invalid demonstration: trend_pct helps define the proxy label.
# This is used only to prove that the audit can detect leakage.
X_leaky_demo = X.copy()
X_leaky_demo["trend_pct_ILLEGAL_DEMO"] = df["trend_pct"]

leaky_numeric_features = numeric_features + ["trend_pct_ILLEGAL_DEMO"]

leaky_model = make_model(leaky_numeric_features)
leaky_model.fit(
    X_leaky_demo.iloc[train_index],
    y.iloc[train_index],
)

leaky_scores = leaky_model.predict_proba(
    X_leaky_demo.iloc[test_index]
)[:, 1]

leaky_p50 = precision_at_k(
    y.iloc[test_index],
    leaky_scores,
    TOP_K,
)

leakage_check = pd.DataFrame([
    {
        "feature set": "Safe final feature set",
        "grouped Precision@50": grouped_p50,
        "valid for starter proxy task": "Yes",
    },
    {
        "feature set": "Invalid demo: adds trend_pct",
        "grouped Precision@50": leaky_p50,
        "valid for starter proxy task": "No — label-derived leakage",
    },
])

display(leakage_check.round(3))

Excluded fields found in safe feature frame: []


,feature set,grouped Precision@50,valid for starter proxy task
0,Safe final feature set,0.84,Yes
1,Invalid demo: adds trend_pct,1.00,No — label-derived leakage


### Leakage verdict

The final feature set contains none of the excluded identifier, metadata, label-definition, or 30-day comparison fields.

The deliberate `trend_pct` demonstration is invalid because `trend_direction`, and therefore the proxy label, is derived from it. Any improved score from that demonstration is evidence that the leakage test works, not a model result to claim.

The grouped split prevents client overlap, but the starter CSV contains trailing-90-day aggregates and a contemporaneous proxy label. Therefore, this notebook does not prove future decline prediction. Future-time validation requires earlier feature windows and a strictly later outcome window from the daily warehouse data.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Bold claim:
The model predicts which pages will decline and identifies pages that should be refreshed.

### Safer claim:
On a grouped held-out set, the logistic-regression model ranked pages with the starter proxy decline label more effectively than the frozen baseline. This observed result can support human review prioritization, but it does not establish future decline prediction or show that refreshing a page will cause recovery.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.